# 3-D Fire & Smoke Evacuation Demo

Multi-floor building with spreading fire and rising smoke.  
The agent starts on the **top floor** and must reach the **ground-floor exit** via staircases.

In [ ]:
import gymnasium as gym
from gymnasium.envs.registration import register
from gymnasium.wrappers import RecordVideo
from environments import FireSmoke3DEnv
from globals import MAX_STEPS, SPEED

register(
    id="FireSmoke3D-v0",
    entry_point="environments:FireSmoke3DEnv",
)

In [ ]:
base_env = gym.make(
    "FireSmoke3D-v0",
    render_mode="rgb_array",
    width=16,
    height=16,
    num_floors=3,
    spread_rate=0.04,
    max_steps=300,
)

env = RecordVideo(
    base_env,
    video_folder="fire_3d_videos",
    episode_trigger=lambda x: True,
    name_prefix="fire_3d_smart_agent",
    fps=5,
)

obs, info = env.reset()
print("Mission:", info["mission"])
print(f"Agent starts at floor {env.unwrapped.agent_pos[2]}")
print(f"Goal is at floor {env.unwrapped.goal_pos[2]}")

In [ ]:
NUM_EPISODES = 2

for episode in range(NUM_EPISODES):
    print(f"\n--- Episode {episode + 1} ---")
    obs, info = env.reset()

    for step in range(300):
        action = env.unwrapped._smart_action_choice()
        obs, reward, terminated, truncated, info = env.step(action)

        pos = env.unwrapped.agent_pos
        print(
            f"  Step {step+1:03d}  action={action}  "
            f"pos=({pos[0]},{pos[1]},F{pos[2]})  "
            f"reward={reward:.1f}  done={terminated}"
        )

        if terminated or truncated:
            print(f"  Episode ended at step {step+1}, reward={reward:.1f}")
            break

env.close()
print("\nDone – videos saved to fire_3d_videos/")

## Optional: 3-D Voxel Snapshot

In [ ]:
import matplotlib.pyplot as plt
from renderer import Renderer3D

test_env = FireSmoke3DEnv(render_mode="rgb_array", width=12, height=12, num_floors=3, spread_rate=0.15)
test_env.reset()

# Run a few steps so fire spreads
for _ in range(20):
    test_env.step(test_env._smart_action_choice())

r = Renderer3D(test_env.grid)
img_3d = r.render_3d(agent_pos=tuple(test_env.agent_pos))

plt.figure(figsize=(10, 8))
plt.imshow(img_3d)
plt.axis("off")
plt.title("3-D Voxel View")
plt.show()